In [1]:
# Import relevant libraries
import os
import numpy as np
import pandas as pd
import dabest
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')

import warnings
warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

# Import summary_ci_1group for individual group mean calculations
from dabest._stats_tools.confint_1group import summary_ci_1group

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:00<00:00, 21.62it/s]

Numba compilation complete!


## FUNCTION

In [2]:
# File paths
officecomp = "C:\\Users\\Star\\"
labcomp = "C:\\Users\\User\\"
computer2 = "C:\\Users\\lnico\\"
homecomp = "D:\\"
specifiedpath = labcomp

# Input directory (compiled OSAR files)
input_dir = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\2025collection\\"

# Output directory for thesis stats
output_dir = specifiedpath + "ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\thesis_stats\\"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# List all files
files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]
print(f"Found {len(files)} files")
print(files[:5])  # Show first 5

Found 82 files
['MB011B x ACR.csv', 'MB011B x Chrimson2.csv', 'MB018B x ACR.csv', 'MB018B x Chrimson2.csv', 'MB077B x ACR.csv']


In [4]:
## Thesis OSAR Function - Creates standardized DataFrames for thesis

def thesis_osar_hedgesg(df, metric_col, df_naming, driver, responder):
    """
    Process OSAR data using hedges_g for each light intensity.
    Returns 8 rows: 4 light intensities x 2 groups (Control, Test)
    """
    try:
        light_intensities = ["Eighth", "Quarter", "Half", "Full"]
        
        # Genotype strings
        genotype_control = f"w1118;;UAS-{responder}/+\nw1118;{driver}/+;{driver}/+"
        genotype_test = f"w1118;{driver}/+;{driver}/{responder}"
        
        rows = []
        
        for intensity in light_intensities:
            # Filter data for this light intensity
            df_intensity = df[df['light_intensity'] == intensity].copy()
            
            # Skip if no data for this intensity
            if len(df_intensity) == 0:
                continue
            
            # Skip if metric column doesn't exist
            if metric_col not in df_intensity.columns:
                continue
            
            # Remove NaN and infinite values for the metric BEFORE passing to dabest
            df_intensity = df_intensity[np.isfinite(df_intensity[metric_col])]
            
            if len(df_intensity) == 0:
                continue
            
            # Separate Sibling (Control) vs Offspring (Test)
            df_control = df_intensity[df_intensity['status'] == 'Sibling']
            df_test = df_intensity[df_intensity['status'] == 'Offspring']
            
            # Skip if either group is empty
            if len(df_control) == 0 or len(df_test) == 0:
                continue
            
            # Prepare data for dabest
            df_intensity['Group'] = df_intensity['status'].map({'Sibling': 'Control', 'Offspring': 'Test'})
            
            # Load dabest for hedges_g comparison
            db = dabest.load(df_intensity, idx=("Control", "Test"), y=metric_col, x='Group')
            results = db.hedges_g.results
            
            # Check if results are valid
            if len(results) == 0:
                continue
            
            # Get plot data for mean calculations
            plot_data = db._plot_data
            xvar = db._xvar
            yvar = db._yvar
            
            # Process Control and Test groups
            groups = ["Control", "Test"]
            genotypes = [genotype_control, genotype_test]
            
            for group, genotype in zip(groups, genotypes):
                # Get group data for mean calculation - ensure it's a proper numpy float array
                group_data = plot_data[plot_data[xvar] == group][yvar].values
                group_data = np.array(group_data, dtype=float)
                
                if len(group_data) == 0:
                    continue
                
                # Filter out any remaining NaN/inf (should already be clean, but just in case)
                group_data = group_data[np.isfinite(group_data)]
                if len(group_data) == 0:
                    continue
                
                # Calculate mean and CI
                group_stats = summary_ci_1group(
                    x=group_data,
                    func=np.mean,
                    resamples=5000,
                    alpha=95,  # Pass 95 for 95% CI (this is actually ci percentage, not alpha)
                    random_seed=12345
                )

                mean_value = round(group_stats['summary'], 2)
                mean_ci_low = round(group_stats['bca_ci_low'], 2)
                mean_ci_high = round(group_stats['bca_ci_high'], 2)
                sample_size = len(group_data)

                # Effect Size - only for Test row (use .iloc for proper indexing)
                if group == "Test":
                    effect_size_value = round(results.difference.iloc[0], 2)
                    effect_size_ci_low = round(results.bca_low.iloc[0], 2)
                    effect_size_ci_high = round(results.bca_high.iloc[0], 2)
                    delta_object = "Hedges' g"
                else:
                    effect_size_value = " "
                    effect_size_ci_low = " "
                    effect_size_ci_high = " "
                    delta_object = " "

                rows.append({
                    "MBON": driver,
                    "Responder": responder,
                    "Light Intensity": intensity,
                    "Group": group,
                    "Genotype": genotype,
                    "Sample Size": sample_size,
                    "Mean": mean_value,
                    "Mean_CI_low": mean_ci_low,
                    "Mean_CI_high": mean_ci_high,
                    "Effect Size": effect_size_value,
                    "Effect_Size_CI_low": effect_size_ci_low,
                    "Effect_Size_CI_high": effect_size_ci_high,
                    "Delta Object": delta_object,
                    "Delta-Delta/Delta-g": " ",  # Always blank for OSAR (no delta2)
                    "Metric": df_naming
                })
        
        return pd.DataFrame(rows)
    
    except Exception as e:
        print(f"  Error in thesis_osar_hedgesg for {driver} - {df_naming}: {e}")
        return pd.DataFrame()

In [5]:
# Process all OSAR files

# Metrics to process (pace renamed to bout)
metrics = [
    ("pi_smoothed_Pattern 01", "pi"),
    ("light_attraction_index_Pattern 01", "light_attraction_index"),
    ("speed_ratio_Pattern 01", "speed_ratio"),
    ("log2_speed_ratio_Pattern 01", "log2_speed_ratio"),
    ("pace_ratio_Pattern 01", "bout_speed_ratio"),  # pace = bout speed
    ("log2_pace_ratio_Pattern 01", "log2_bout_speed_ratio"),  # pace = bout speed
    ("bout_index_Pattern 01", "bout_number_index"),
    ("bout_duration_ratio_Pattern 01", "bout_duration_ratio"),
    ("max_velocity_ratio_Pattern 01", "max_velocity_ratio"),
]

# MBONs to exclude
excluded_mbons = ["R76B09"]

# Store all results
all_thesis_dfs = []

for file in files:
    # Parse filename to get driver and responder
    # Format: "MB112C x ACR.csv" -> driver="MB112C", responder="ACR"
    filename_no_ext = file.replace('.csv', '')
    parts = filename_no_ext.split(' x ')
    
    if len(parts) != 2:
        print(f"Skipping file with unexpected format: {file}")
        continue
    
    driver = parts[0].strip()
    responder = parts[1].strip()
    
    # Skip excluded MBONs
    if driver in excluded_mbons:
        print(f"Skipping excluded MBON: {driver} x {responder}")
        continue
    
    print(f"Processing: {driver} x {responder}")
    
    # Read the CSV file
    df = pd.read_csv(input_dir + file)
    
    # Process each metric
    driver_dfs = []
    for metric_col, df_naming in metrics:
        try:
            metric_df = thesis_osar_hedgesg(df, metric_col, df_naming, driver, responder)
            if len(metric_df) > 0:
                driver_dfs.append(metric_df)
        except Exception as e:
            print(f"  Error processing {df_naming} for {driver}: {e}")
    
    # Concatenate all metrics for this driver
    if driver_dfs:
        driver_thesis_df = pd.concat(driver_dfs, ignore_index=True)
        all_thesis_dfs.append(driver_thesis_df)
        
        # Save individual driver file
        driver_thesis_df.to_csv(output_dir + f"{driver} x {responder}_thesis_stats.csv", index=False)

print("\nDone processing all files!")

Processing: MB011B x ACR
Processing: MB011B x Chrimson2
Processing: MB018B x ACR
Processing: MB018B x Chrimson2
Processing: MB077B x ACR
Processing: MB077B x Chrimson2
Processing: MB080C x ACR
Processing: MB080C x Chrimson2
Processing: MB082C x ACR
Processing: MB082C x Chrimson2
Processing: MB083C x ACR
Processing: MB083C x Chrimson2
Processing: MB093C x ACR
Processing: MB093C x Chrimson2
Processing: MB112C x ACR
Processing: MB112C x Chrimson2
Processing: MB210B x ACR
Processing: MB210B x Chrimson2
Processing: MB242A x ACR
Processing: MB242A x Chrimson2
Processing: MB310C x ACR
Processing: MB310C x Chrimson2
Processing: MB319C x ACR
Processing: MB319C x Chrimson2
Processing: MB323B x ACR
Processing: MB323B x Chrimson2
Processing: MB434B x ACR
Processing: MB434B x Chrimson2
Processing: MB542B x ACR
Processing: MB542B x Chrimson2
Skipping excluded MBON: R76B09 x ACR
Skipping excluded MBON: R76B09 x Chrimson2
Processing: SS01188 x ACR
Processing: SS01188 x Chrimson2
Processing: SS01194 x 

## Total file

In [7]:
df_osar_thesis_all = pd.DataFrame()
individualfiles = "D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\thesis_stats\\"
indi_files = os.listdir(individualfiles)
for j in indi_files:
    all_thesis_dfs = pd.read_csv(individualfiles + j)
    df_osar_thesis_all = pd.concat([df_osar_thesis_all, all_thesis_dfs], ignore_index=True)
    
df_osar_thesis_all.to_csv(output_dir + f"{date}_all_osar_thesis_stats.csv", index=False)

## total file - if ran individual files


In [17]:
inputdir_individual = "D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\thesis_stats\\"
output_dir_individual = "D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\osar_compiled\\"
files_individual = [f for f in os.listdir(inputdir_individual) if f.endswith('_thesis_stats.csv')]
df_osar_thesis_all = pd.DataFrame()

for j in files_individual:
    df_osar_thesis_all = pd.concat([df_osar_thesis_all, pd.read_csv(inputdir_individual + j)], ignore_index=True)


df_osar_thesis_all.to_csv(output_dir_individual + f"{date}_all_osar_thesis_stats.csv", index=False)